## **Aim**
To implement a program that detects suspicious file extensions, including double-extension files such as invoice.pdf.exe.

## **Algorithm**
**Step 1:** Import `os`, `re`, and `collections` libraries.

**Step 2:** Define a list of known dangerous extensions (e.g., .exe, .bat, .scr, .vbs, .js, .jar, .ps1, .cmd, .com, .pif).

**Step 3:** Define a list of common document extensions that are often spoofed (e.g., .pdf, .doc, .docx, .xls, .xlsx, .txt, .rtf).

**Step 4:** Create a function `analyze_filename(filename)` that:
   - Extracts the full extension (everything after the last dot)
   - Checks for double/multiple extensions (e.g., .pdf.exe)
   - Checks if the final extension is in the dangerous list
   - Checks if a document extension is followed by a dangerous extension

**Step 5:** Assign a risk score based on findings.

**Step 6:** Scan a directory recursively and report suspicious files.

In [1]:
import os
import re
import shutil
from datetime import datetime

DANGEROUS_EXTS = {
    '.exe', '.bat', '.cmd', '.com', '.scr', '.vbs', '.vbe',
    '.js', '.jse', '.wsf', '.wsh', '.ps1', '.ps1xml', '.ps2',
    '.ps2xml', '.psc1', '.psc2', '.msh', '.msh1', '.msh2',
    '.mshxml', '.msh1xml', '.msh2xml', '.jar', '.py', '.pl',
    '.sh', '.bash', '.csh', '.ksh', '.tcl', '.reg', '.lnk',
    '.inf', '.ins', '.isp', '.msp', '.mst', '.pif', '.application',
    '.gadget', '.msi', '.msp', '.hta', '.cpl', '.msc', '.jar'
}

DOCUMENT_EXTS = {
    '.pdf', '.doc', '.docx', '.xls', '.xlsx', '.ppt', '.pptx',
    '.txt', '.rtf', '.odt', '.ods', '.odp', '.csv', '.xml',
    '.html', '.htm', '.jpg', '.jpeg', '.png', '.gif', '.bmp',
    '.tiff', '.tif', '.mp3', '.mp4', '.avi', '.mov', '.zip',
    '.rar', '.7z', '.tar', '.gz'
}

def analyze_filename(filename):
    # Get all extensions (parts after dots)
    parts = filename.split('.')
    if len(parts) < 2:
        return {"risk": "LOW", "score": 0, "issues": [], "final_ext": "", "double_ext": False}
    
    extensions = ['.' + p.lower() for p in parts[1:]]
    final_ext = extensions[-1]
    issues = []
    score = 0
    
    # Check for double extension (document + executable)
    if len(extensions) >= 2:
        for i in range(len(extensions) - 1):
            if extensions[i] in DOCUMENT_EXTS and extensions[i+1] in DANGEROUS_EXTS:
                issues.append(f"Double extension: {extensions[i]}{extensions[i+1]} (document + executable)")
                score += 50
    
    # Check if final extension is dangerous
    if final_ext in DANGEROUS_EXTS:
        issues.append(f"Dangerous executable extension: {final_ext}")
        score += 40
    
    # Check for hidden extensions (many dots)
    if len(extensions) > 2:
        issues.append(f"Multiple extensions ({len(extensions)}): {'.'.join(extensions)}")
        score += 10
    
    # Check for right-to-left override or other unicode tricks
    if '\u202e' in filename or '\u202d' in filename:
        issues.append("Unicode direction override character detected")
        score += 30
    
    # Determine risk level
    if score >= 50:
        risk = "CRITICAL"
    elif score >= 30:
        risk = "HIGH"
    elif score >= 15:
        risk = "MEDIUM"
    elif score > 0:
        risk = "LOW"
    else:
        risk = "SAFE"
    
    return {
        "risk": risk,
        "score": score,
        "issues": issues,
        "final_ext": final_ext,
        "all_extensions": extensions,
        "double_ext": len(extensions) >= 2 and any(e in DOCUMENT_EXTS for e in extensions[:-1]) and extensions[-1] in DANGEROUS_EXTS
    }

def scan_directory(directory):
    results = []
    for root, _, files in os.walk(directory):
        for file in files:
            analysis = analyze_filename(file)
            if analysis["score"] > 0:
                filepath = os.path.join(root, file)
                stat = os.stat(filepath)
                results.append({
                    "file": file,
                    "path": filepath,
                    "size": stat.st_size,
                    "analysis": analysis
                })
    return results

def main():
    test_dir = "./suspicious_ext_test"
    
    if os.path.exists(test_dir):
        shutil.rmtree(test_dir)
    os.makedirs(test_dir)
    os.makedirs(os.path.join(test_dir, "documents"))
    os.makedirs(os.path.join(test_dir, "downloads"))
    
    # Create test files with various extensions
    test_files = [
        "normal_document.pdf",
        "invoice.pdf.exe",
        "report.docx.scr",
        "photo.jpg",
        "script.bat",
        "malware.exe",
        "data.txt.vbs",
        "presentation.pptx.js",
        "archive.zip.exe",
        "readme.txt",
        "setup.msi",
        "evidence\u202efdp.exe"  # RTL override: evidence.exe.pdf
    ]
    
    for fname in test_files:
        with open(os.path.join(test_dir, fname), "w") as f:
            f.write("test")
    
    print(f"Scanning for suspicious extensions in: {test_dir}\n")
    print(f"{'File Name':<30} {'Risk':<10} {'Score':<6} Issues")
    print("-" * 80)
    
    results = scan_directory(test_dir)
    for r in sorted(results, key=lambda x: -x["analysis"]["score"]):
        issues_str = "; ".join(r["analysis"]["issues"]) if r["analysis"]["issues"] else "None"
        print(f"{r['file']:<30} {r['analysis']['risk']:<10} {r['analysis']['score']:<6} {issues_str}")

if __name__ == "__main__":
    main()

Scanning for suspicious extensions in: ./suspicious_ext_test

File Name                      Risk       Score  Issues
--------------------------------------------------------------------------------
invoice.pdf.exe                CRITICAL   90     Double extension: .pdf.exe (document + executable); Dangerous executable extension: .exe
report.docx.scr                CRITICAL   90     Double extension: .docx.scr (document + executable); Dangerous executable extension: .scr
data.txt.vbs                   CRITICAL   90     Double extension: .txt.vbs (document + executable); Dangerous executable extension: .vbs
presentation.pptx.js           CRITICAL   90     Double extension: .pptx.js (document + executable); Dangerous executable extension: .js
archive.zip.exe                CRITICAL   90     Double extension: .zip.exe (document + executable); Dangerous executable extension: .exe
evidence‮fdp.exe               CRITICAL   70     Dangerous executable extension: .exe; Unicode direction overri

## **Result**
This the program successfully detects suspicious file extensions, including double-extension files such as invoice.pdf.exe.